**Imports**

In [1]:
import os
import yaml
from ultralytics import YOLO

ModuleNotFoundError: No module named 'ultralytics'

**Create YAML**

In [ ]:
dataset = {
    "path": os.path.abspath("YTU-CrackIS"),   #Path to dataset
    "train": "images/train2017",   # ← added (We just need it to exist. It is not used during prediction)
    "val":   "images/val2017",
    "nc":    1,
    "names": ["crack"]
}

yaml_path = "crack_dataset.yaml"
with open(yaml_path, "w") as f:
    yaml.dump(dataset, f, default_flow_style=False, sort_keys=False)

print("✅ dataset.yaml created:")
with open(yaml_path) as f:
    print(f.read())

**Load Model & Print Summary**

In [ ]:
model = YOLO("8x-608-l4/weights/last.pt", task="segment") #Path to weights file
print(f"✅ Model loaded")
print(f"   Task    : {model.task}")
print(f"   Classes : {model.names}")

**Run Validation & Calculate Metrics**

In [ ]:
print("\n⏳ Running validation — this may take a few minutes...\n")
results = model.val(
    data=yaml_path,
    split="val",
    imgsz=640,     
    batch=4,
    device='cpu',        # use GPU if available; change to 'cpu' if not
    verbose=True,
    plots=True,      # saves confusion matrix, PR curve, etc.
)

# ── 4. Print key metrics ───────────────────────────────────────────────────
print("\n── Results ────────────────────────────────────────────────────────")
print(f"  Box  mAP@0.5      : {results.box.map50:.4f}")
print(f"  Box  mAP@0.5:0.95 : {results.box.map:.4f}")
print(f"  Mask mAP@0.5      : {results.seg.map50:.4f}")
print(f"  Mask mAP@0.5:0.95 : {results.seg.map:.4f}")
print(f"\n✅ Plots saved to: {results.save_dir}")

**Alternate: If you want to just run prediction on images in a folder**

In [ ]:
!yolo task=segment mode=predict model=8x-608-l4/weights/last.pt  conf=0.20 source=test save=True show_boxes=false